In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import numpy as np

# RandomForest Model

In [23]:
file_path = 'datasets/Preprocessed_csv.csv'
data = pd.read_csv(file_path)
data.head()

,Date,clean_text,favorite_count,full_text,reply_count,retweet_count,entity_vectors,clean_text_vector,importance_coefficient,importance_coefficient_normalized,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
0,2021-02-01,sent givedirectly great work distributing fund...,3496.0,i sent some! https://t.co/mfyrz35zjf\n\nyou sh...,731.0,686,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[ 6.55804798e-02 -4.08827849e-02 5.29426113e-...,8043.5,0.013905,...,34638.21,32384.23,33537.18,33537.18,61400400660,0.01,0.00,-9.08,-2.39,1
1,2021-02-02,watch video learn truth doublespend separate f...,109.0,watch this video to learn the truth about what...,5.0,25,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[ 4.15213443e-02 -1.58386230e-02 -1.94963720e-...,245.5,0.000424,...,35896.88,33489.22,35510.29,35510.29,63088585433,0.06,0.02,1973.11,-2.39,1
2,2021-02-03,min video review strategy simple advanced help...,153.0,"in this 7min video, i review strategies from s...",9.0,24,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[-0.0159668 0.06502838 -0.05678711 0.042077...,334.5,0.000578,...,37480.19,35443.98,37472.09,37472.09,61166818159,0.06,0.02,1961.80,-11.31,1
3,2021-02-04,fair elon gave chance load bag early,196.0,to be fair elon gave all of you the chance to ...,31.0,18,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[ 9.04134139e-02 9.13085938e-02 1.20035803e-...,425.5,0.000736,...,38592.18,36317.50,36926.07,36926.07,68838074392,-0.01,0.03,-546.02,-2507.82,0
4,2021-02-05,minute video ive learned much cant smash like ...,100.0,â30 minutes into the video and i've learned ...,7.0,8,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[ 0.02418934 0.05538386 -0.01659463 0.096178...,211.5,0.000366,...,38225.91,36658.76,38144.31,38144.31,58598066402,0.03,0.03,1218.24,1764.27,1


### Data Preparation

In [24]:
features = [
    'compound', 'positive_sentiment_score', 'negative_sentiment_score', 'neutral_sentiment_score',
    'favorite_count', 'reply_count', 'retweet_count', 'importance_coefficient', 'importance_coefficient_normalized',
    'Open', 'Volume'
]

target = 'target'

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

((673, 11), (169, 11), (673,), (169,))

### Model training

In [25]:
rf_classifier = RandomForestClassifier(random_state=42)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)
initial_accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)
print(f'Initial Model Accuracy: {round(initial_accuracy*100, 2)}%\n')
print(classification_rep)

Initial Model Accuracy: 48.52%

              precision    recall  f1-score   support

           0       0.57      0.46      0.51        99
           1       0.40      0.51      0.45        70

    accuracy                           0.49       169
   macro avg       0.49      0.49      0.48       169
weighted avg       0.50      0.49      0.49       169


### Model Evaluation and Refinement - Examining Feature Importances

In [26]:
feature_importances = rf_classifier.feature_importances_
features_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
features_df = features_df.sort_values(by='Importance', ascending=False)
features_df

,Feature,Importance
10,Volume,0.123003
9,Open,0.112345
5,reply_count,0.106192
6,retweet_count,0.098330
7,importance_coefficient,0.094225
8,importance_coefficient_normalized,0.092876
0,compound,0.091449
4,favorite_count,0.091211
3,neutral_sentiment_score,0.082646
1,positive_sentiment_score,0.064990


### Hyperparameter Tuning

In [27]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=rf_classifier, param_grid=param_grid, 
                           cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

grid_search.fit(X_train, y_train)
best_score = grid_search.best_score_
print(f'\nBest Score: {round(best_score*100, 2)}%')

best_params = grid_search.best_params_

best_params

Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Score: 54.09%


{'max_depth': 10,
 'min_samples_leaf': 2,
 'min_samples_split': 10,
 'n_estimators': 200}

### Retraining the RandomForest model with the best hyperparameters from the grid search

In [28]:
rf_classifier_best = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=2,
    random_state=42
)

rf_classifier_best.fit(X_train, y_train)
y_pred_best = rf_classifier_best.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
classification_rep_best = classification_report(y_test, y_pred_best)
print(f'Accuracy: {round(accuracy_best*100, 2)}%\n')
print(classification_rep_best)

Accuracy: 49.11%

              precision    recall  f1-score   support

           0       0.59      0.44      0.51        99
           1       0.41      0.56      0.48        70

    accuracy                           0.49       169
   macro avg       0.50      0.50      0.49       169
weighted avg       0.52      0.49      0.49       169


### Cross-validation

In [29]:
cv_scores = cross_val_score(rf_classifier_best, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f'Cross-validation Mean score: {round(cv_mean*100, 2)}%')
print(f'Cross-validation Standard Deviation score: {round(cv_mean*100, 2)}%')
print('Cross-validation Individual Fold Accuracies:')
i = 1
for sc in cv_scores:
   print(f' - Fold {i}: {round(sc*100, 2)}%')
   i += 1

Cross-validation Mean score: 52.97%
Cross-validation Standard Deviation score: 52.97%
Cross-validation Individual Fold Accuracies:
 - Fold 1: 50.3%
 - Fold 2: 57.99%
 - Fold 3: 54.76%
 - Fold 4: 51.19%
 - Fold 5: 50.6%


### Vector Extraction

In [30]:
def convert_vector(row):
    return np.array([float(x) for x in row.strip('[]').split()])
data['clean_text_vector'] = data['clean_text_vector'].apply(convert_vector)


### Data Preparation

In [31]:
vector_features = np.stack(data['clean_text_vector'].values)

X_combined = np.concatenate([X, vector_features], axis=1)

X_train_combined, X_test_combined, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

(X_train_combined.shape, X_test_combined.shape, y_train.shape, y_test.shape)

((673, 311), (169, 311), (673,), (169,))

### Model Integration - Retraining RandomForest with the combined dataset

In [32]:
# Initialize the RandomForest classifier with the previously identified best hyperparameters
rf_classifier_combined = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=2,
    random_state=42
)

rf_classifier_combined.fit(X_train_combined, y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=2, n_estimators=50,
                       random_state=42)

### Model Evaluation

In [33]:
y_pred_combined = rf_classifier_combined.predict(X_test_combined)
accuracy_combined = accuracy_score(y_test, y_pred_combined)
classification_rep_combined = classification_report(y_test, y_pred_combined)

print(f'Accuracy: {round(accuracy_combined*100, 2)}%\n')
print(classification_rep_combined)

Accuracy: 55.03%

              precision    recall  f1-score   support

           0       0.64      0.55      0.59        99
           1       0.46      0.56      0.51        70

    accuracy                           0.55       169
   macro avg       0.55      0.55      0.55       169
weighted avg       0.56      0.55      0.55       169


### Model Evaluation and Refinement - Examining Feature Importances

In [34]:
# Extract Feature Importances from the RandomForest model
feature_importances_combined = rf_classifier_combined.feature_importances_
original_feature_names = X.columns.tolist()
vector_feature_names = [f'vector_{i}' for i in range(vector_features.shape[1])]
all_feature_names = original_feature_names + vector_feature_names
features_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': feature_importances_combined
})

sorted_features = features_importance_df.sort_values(by='Importance', ascending=False)

sorted_features.head(10)  

,Feature,Importance
225,vector_214,0.007406
35,vector_24,0.007147
288,vector_277,0.006829
103,vector_92,0.006504
124,vector_113,0.006416
219,vector_208,0.006076
156,vector_145,0.006068
139,vector_128,0.006002
99,vector_88,0.005997
111,vector_100,0.005981


### Originaly used features performance among the tweet content

In [35]:
original_features_positions = sorted_features.reset_index()
original_features_positions['Rank'] = original_features_positions.index + 1

original_features_ranking = original_features_positions[
    original_features_positions['Feature'].isin(original_feature_names)
]

original_features_ranking = original_features_ranking.sort_values('Rank')
original_features_ranking

,index,Feature,Importance,Rank
19,3,neutral_sentiment_score,0.005257,20
38,7,importance_coefficient,0.004714,39
68,6,retweet_count,0.004167,69
91,10,Volume,0.003777,92
101,5,reply_count,0.003658,102
139,8,importance_coefficient_normalized,0.003285,140
144,9,Open,0.003262,145
146,4,favorite_count,0.003243,147
180,2,negative_sentiment_score,0.002882,181
255,0,compound,0.002005,256


### Selecting the best-performing original features and the top of the most important tweet content features

In [36]:
num_top_original_features = len(features_df) // 2 # Selecting the top half of the original features

top_original_features = sorted_features[
    ~sorted_features['Feature'].str.startswith('vector_')
].head(num_top_original_features)['Feature']

X_top_original = data[top_original_features]

num_tweet_features = len(vector_feature_names) // 4
top_vector_feature_labels = sorted_features[sorted_features['Feature'].str.startswith('vector_')].head(num_tweet_features)['Feature']

top_vector_indices = [int(label.split('_')[1]) for label in top_vector_feature_labels]

top_vector_features = vector_features[:, top_vector_indices]

X_top_combined = pd.concat([
    X_top_original.reset_index(drop=True),
    pd.DataFrame(top_vector_features, columns=top_vector_feature_labels)
], axis=1)

X_top_combined.head()

,neutral_sentiment_score,importance_coefficient,retweet_count,Volume,reply_count,vector_214,vector_24,vector_277,vector_92,vector_113,...,vector_243,vector_220,vector_133,vector_21,vector_132,vector_189,vector_211,vector_44,vector_90,vector_31
0,1.000,8043.5,686,61400400660,731.0,-0.081001,0.056577,-0.122192,-0.039572,-0.024267,...,-0.046757,-0.063349,0.014185,0.096464,-0.053733,0.073845,0.048757,0.024615,0.080243,-0.015474
1,0.925,245.5,25,63088585433,5.0,-0.096017,-0.023743,-0.084111,-0.063232,0.052665,...,-0.124390,0.023045,0.033491,0.125117,-0.045271,0.011335,0.200806,-0.094535,0.010577,0.118722
2,0.864,334.5,24,61166818159,9.0,-0.044202,-0.031329,-0.143921,-0.047583,-0.037543,...,-0.109985,-0.043793,0.014639,0.062849,-0.042621,-0.042542,0.050464,-0.059192,0.088477,-0.012341
3,0.902,425.5,18,68838074392,31.0,-0.031860,0.110474,-0.018738,-0.026993,-0.029460,...,-0.213542,0.033081,-0.023153,-0.013316,-0.100586,0.077443,0.174357,-0.061442,0.046885,0.002808
4,0.705,211.5,8,58598066402,7.0,-0.068210,0.016688,-0.119421,-0.068186,0.022835,...,-0.155296,-0.047058,0.012471,0.095833,-0.041079,0.000689,0.083346,0.032798,0.037448,0.075309


### Model Integration - Retraining RandomForest with the best-performing features

In [37]:
X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top_combined, y, test_size=0.2, random_state=42
)

rf_classifier_top = RandomForestClassifier(random_state=42)
rf_classifier_top.fit(X_train_top, y_train_top)

RandomForestClassifier(random_state=42)

### Model Evaluation

In [38]:
y_pred_top = rf_classifier_top.predict(X_test_top)
accuracy_top = accuracy_score(y_test_top, y_pred_top)
classification_rep_top = classification_report(y_test_top, y_pred_top)

print(f'Accuracy: {round(accuracy_top*100, 2)}%')
print(classification_rep_top)

Accuracy: 53.25%
              precision    recall  f1-score   support

           0       0.61      0.55      0.58        99
           1       0.44      0.51      0.48        70

    accuracy                           0.53       169
   macro avg       0.53      0.53      0.53       169
weighted avg       0.54      0.53      0.54       169


### Model Evaluation - Examining Feature Importances

In [39]:
feature_importances_top_combined = rf_classifier_top.feature_importances_

all_top_feature_names = X_top_combined.columns.tolist()

features_importance_df = pd.DataFrame({
    'Feature': all_top_feature_names,
    'Importance': feature_importances_top_combined
})


sorted_features = features_importance_df.sort_values(by='Importance', ascending=False)

num_top = len(all_top_feature_names) // 2
sorted_features.head(num_top)  # Display the top half most important features

,Feature,Importance
47,vector_1,0.017717
39,vector_286,0.016885
31,vector_167,0.016823
10,vector_208,0.016791
62,vector_7,0.015621
24,vector_150,0.015602
4,reply_count,0.015539
14,vector_100,0.015497
19,vector_110,0.015424
7,vector_277,0.015334


### Tweet Related Features Testing

#### Testing the model with all the tweet-related features

In [40]:
tweet_related_features = [
    'compound', 'positive_sentiment_score', 'negative_sentiment_score', 'neutral_sentiment_score',
    'favorite_count', 'reply_count', 'retweet_count', 'importance_coefficient', 'importance_coefficient_normalized',
]

X_tweet_related = data[tweet_related_features]

X_tweet_related_combined = np.concatenate([X_tweet_related, vector_features], axis=1)

X_train_tweet_only, X_test_tweet_only, y_train_tweet_only, y_test_tweet_only = train_test_split(
    X_tweet_related_combined, y, test_size=0.2, random_state=42
)

rf_classifier_tweet_only = RandomForestClassifier(random_state=42)
rf_classifier_tweet_only.fit(X_train_tweet_only, y_train_tweet_only)

y_pred_tweet_only = rf_classifier_tweet_only.predict(X_test_tweet_only)
accuracy_tweet_only = accuracy_score(y_test_tweet_only, y_pred_tweet_only)
classification_rep_tweet_only = classification_report(y_test_tweet_only, y_pred_tweet_only)

print(f'Accuracy (Tweet Only): {accuracy_tweet_only*100:.2f}%')
print(classification_rep_tweet_only)

Accuracy (Tweet Only): 53.85%
              precision    recall  f1-score   support

           0       0.62      0.55      0.58        99
           1       0.45      0.53      0.49        70

    accuracy                           0.54       169
   macro avg       0.54      0.54      0.53       169
weighted avg       0.55      0.54      0.54       169


#### Testing the model with only the tweet content

In [41]:
X_tweet_content = vector_features

X_train_tweet_content, X_test_tweet_content, y_train_tweet_content, y_test_tweet_content = train_test_split(
    X_tweet_content, y, test_size=0.2, random_state=42
)

rf_classifier_tweet_content = RandomForestClassifier(random_state=42)
rf_classifier_tweet_content.fit(X_train_tweet_content, y_train_tweet_content)

y_pred_tweet_content = rf_classifier_tweet_content.predict(X_test_tweet_content)
accuracy_tweet_content = accuracy_score(y_test_tweet_content, y_pred_tweet_content)
classification_rep_tweet_content = classification_report(y_test_tweet_content, y_pred_tweet_content)

print(f'Accuracy (Tweet Content): {accuracy_tweet_content*100:.2f}%')
print(classification_rep_tweet_content)


Accuracy (Tweet Content): 51.48%
              precision    recall  f1-score   support

           0       0.60      0.53      0.56        99
           1       0.43      0.50      0.46        70

    accuracy                           0.51       169
   macro avg       0.51      0.51      0.51       169
weighted avg       0.53      0.51      0.52       169


#### Testing the model with only the tweet-related features

In [42]:
X_tweet_related_only = data[tweet_related_features]

X_train_tweet_related_only, X_test_tweet_related_only, y_train_tweet_related_only, y_test_tweet_related_only = train_test_split(
    X_tweet_related_only, y, test_size=0.2, random_state=42
)

rf_classifier_tweet_related_only = RandomForestClassifier(random_state=42)
rf_classifier_tweet_related_only.fit(X_train_tweet_related_only, y_train_tweet_related_only)

y_pred_tweet_related_only = rf_classifier_tweet_related_only.predict(X_test_tweet_related_only)
accuracy_tweet_related_only = accuracy_score(y_test_tweet_related_only, y_pred_tweet_related_only)
classification_rep_tweet_related_only = classification_report(y_test_tweet_related_only, y_pred_tweet_related_only)

print(f'Accuracy (Tweet Related Only): {accuracy_tweet_related_only*100:.2f}%')
print(classification_rep_tweet_related_only)

Accuracy (Tweet Related Only): 49.70%
              precision    recall  f1-score   support

           0       0.58      0.53      0.55        99
           1       0.41      0.46      0.43        70

    accuracy                           0.50       169
   macro avg       0.49      0.49      0.49       169
weighted avg       0.51      0.50      0.50       169


### Market Related Features Testing

In [43]:
market_related_features = [
    'Open', 'Volume', 'historical_volatility'
]

X_market_related_only = data[market_related_features]

X_train_market_related_only, X_test_market_related_only, y_train_market_related_only, y_test_market_related_only = train_test_split(
    X_market_related_only, y, test_size=0.2, random_state=42
)

rf_classifier_market_related_only = RandomForestClassifier(random_state=42)
rf_classifier_market_related_only.fit(X_train_market_related_only, y_train_market_related_only)

y_pred_market_related_only = rf_classifier_market_related_only.predict(X_test_market_related_only)
accuracy_market_related_only = accuracy_score(y_test_market_related_only, y_pred_market_related_only)
classification_rep_market_related_only = classification_report(y_test_market_related_only, y_pred_market_related_only)

print(f'Accuracy (Market Related Only): {accuracy_market_related_only*100:.2f}%')
print(classification_rep_market_related_only)

Accuracy (Market Related Only): 56.80%
              precision    recall  f1-score   support

           0       0.65      0.58      0.61        99
           1       0.48      0.56      0.52        70

    accuracy                           0.57       169
   macro avg       0.56      0.57      0.56       169
weighted avg       0.58      0.57      0.57       169
